In [37]:
# -*- coding: utf-8 -*-
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

# ======================================================
# 🔹 Função que gera o gráfico (Plotly)
# ======================================================
def make_nash_plot(U, d_i, d_j):
    if d_i + d_j >= U:
        fig = go.Figure()
        fig.update_layout(
            title="⚠️ Conjunto inviável: d_i + d_j > U",
            template="plotly_white", 
            width=700, 
            height=600
        )
        return fig

    # Solução de Nash
    U_i = 0.5 * (U + d_i - d_j)
    U_j = 0.5 * (U + d_j - d_i)
    w_i, w_j = U_i / U, U_j / U

    # Fronteira de Pareto
    Ui = np.linspace(d_i, U - d_j, 200)
    Uj = U - Ui

    fig = go.Figure()

    # Linha de Pareto
    fig.add_trace(go.Scatter(
        x=Ui, y=Uj, mode="lines", 
        line=dict(color="red", width=3), 
        name="Fronteira de Pareto"
    ))
    
    # Desacordo
    fig.add_trace(go.Scatter(
        x=[d_i], y=[d_j], mode="markers", 
        marker=dict(color="black", size=10), 
        name=f"Desacordo ({d_i}, {d_j})"
    ))
    
    # Nash
    fig.add_trace(go.Scatter(
        x=[U_i], y=[U_j], mode="markers+text",
        text=[f"Nash ({U_i:.1f}, {U_j:.1f})"],
        textposition="top right",
        marker=dict(color="blue", size=12), 
        name="Solução de Nash"
    ))
    
    # Linha de ganhos iguais
    fig.add_trace(go.Scatter(
        x=[d_i, U_i], y=[d_j, U_j],
        mode="lines", 
        line=dict(color="blue", dash="dash", width=2), 
        name="Linha de ganhos iguais"
    ))

    fig.update_layout(
        title=(
            f"<b>Solução de Nash para Barganha</b><br>"
            f"<sup>U_i* = {U_i:.2f}, U_j* = {U_j:.2f}, w_i = {w_i:.2f}, w_j = {w_j:.2f}</sup>"
        ),
        xaxis_title="Jogador i - Utilidade (MM$)",
        yaxis_title="Jogador j - Utilidade (MM$)",
        xaxis=dict(range=[0, max(U, U_i + 10)], zeroline=True, zerolinewidth=2),
        yaxis=dict(range=[0, max(U, U_j + 10)], zeroline=True, zerolinewidth=2),
        template="plotly_white",
        width=700, 
        height=400,
        showlegend=True
    )
    
    # Adicionar grid
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')
    
    return fig

# ======================================================
# 🔹 Sliders com estilo melhorado
# ======================================================
U_slider = widgets.FloatSlider(
    value=100, min=50, max=200, step=5, 
    description="U Total:", 
    style={'description_width': 'initial'}
)

di_slider = widgets.FloatSlider(
    value=40, min=0, max=100, step=5, 
    description="d_i (Firma 1):", 
    style={'description_width': 'initial'}
)

dj_slider = widgets.FloatSlider(
    value=0, min=0, max=100, step=5, 
    description="d_j (Firma 2):", 
    style={'description_width': 'initial'}
)

# ======================================================
# 🔹 Output interativo
# ======================================================
graph_output = widgets.Output()

# ======================================================
# 🔹 Função de atualização
# ======================================================
def update_plot(change=None):
    with graph_output:
        graph_output.clear_output(wait=True)
        fig = make_nash_plot(U_slider.value, di_slider.value, dj_slider.value)
        display(fig)

# Conectar os sliders à função de atualização
U_slider.observe(update_plot, names='value')
di_slider.observe(update_plot, names='value')
dj_slider.observe(update_plot, names='value')

# ======================================================
# 🔹 Layout da interface
# ======================================================
sliders_box = widgets.VBox([
    widgets.HTML("<h3>Controles Interativos</h3>"),
    U_slider,
    di_slider, 
    dj_slider,
    widgets.HTML("<p><i>Altere os sliders para atualizar o gráfico</i></p>")
])

# ======================================================
# 🔹 Exibir interface
# ======================================================
display(widgets.VBox([
    widgets.HTML("<h1>Modelo de Barganha de Nash</h1>"),
    widgets.HBox([sliders_box, graph_output])
]))

# Gerar gráfico inicial
update_plot()

In [40]:
# -*- coding: utf-8 -*-
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import ipywidgets as widgets
from IPython.display import display

# ======================================================
# 🔹 Configuração do Plotly para melhor performance
# ======================================================
pio.templates.default = "plotly_white"

# ======================================================
# 🔹 Criar FigureWidget inicial
# ======================================================
fig = go.FigureWidget()
fig.update_layout(
    width=700, 
    height=600,
    template="plotly_white",
    xaxis_title="Jogador i - Utilidade (MM$)",
    yaxis_title="Jogador j - Utilidade (MM$)",
    showlegend=True
)

# ======================================================
# 🔹 Função que atualiza o gráfico
# ======================================================
def update_plot(U, d_i, d_j):
    with fig.batch_update():
        fig.data = []  # Limpar todos os traces
        
        if d_i + d_j >= U:
            fig.update_layout(
                title="⚠️ Conjunto inviável: d_i + d_j ≥ U",
                xaxis=dict(range=[0, U], zeroline=True, zerolinewidth=2),
                yaxis=dict(range=[0, U], zeroline=True, zerolinewidth=2)
            )
            # Adicionar apenas um trace vazio para manter o layout
            fig.add_trace(go.Scatter(x=[], y=[], mode="markers"))
            return
        
        # Solução de Nash
        U_i = 0.5 * (U + d_i - d_j)
        U_j = 0.5 * (U + d_j - d_i)
        w_i, w_j = U_i / U, U_j / U

        # Fronteira de Pareto
        Ui = np.linspace(d_i, U - d_j, 200)
        Uj = U - Ui

        # Linha de Pareto
        fig.add_trace(go.Scatter(
            x=Ui, y=Uj, mode="lines", 
            line=dict(color="red", width=3), 
            name="Fronteira de Pareto"
        ))
        
        # Desacordo
        fig.add_trace(go.Scatter(
            x=[d_i], y=[d_j], mode="markers", 
            marker=dict(color="black", size=10), 
            name=f"Desacordo ({d_i}, {d_j})"
        ))
        
        # Nash
        fig.add_trace(go.Scatter(
            x=[U_i], y=[U_j], mode="markers+text",
            text=[f"Nash ({U_i:.1f}, {U_j:.1f})"],
            textposition="top right",
            marker=dict(color="blue", size=12), 
            name="Solução de Nash"
        ))
        
        # Linha de ganhos iguais
        fig.add_trace(go.Scatter(
            x=[d_i, U_i], y=[d_j, U_j],
            mode="lines", 
            line=dict(color="blue", dash="dash", width=2), 
            name="Linha de ganhos iguais"
        ))

        fig.update_layout(
            title=(
                f"<b>Solução de Nash para Barganha</b><br>"
                f"<sup>U_i* = {U_i:.2f}, U_j* = {U_j:.2f}, w_i = {w_i:.2f}, w_j = {w_j:.2f}</sup>"
            ),
            xaxis=dict(range=[0, max(U, U_i + 10)], zeroline=True, zerolinewidth=2),
            yaxis=dict(range=[0, max(U, U_j + 10)], zeroline=True, zerolinewidth=2)
        )
        
        # Adicionar grid
        fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')
        fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')

# ======================================================
# 🔹 Sliders com estilo melhorado
# ======================================================
U_slider = widgets.FloatSlider(
    value=100, min=50, max=200, step=5, 
    description="U Total:", 
    style={'description_width': 'initial'},
    continuous_update=True  # Atualização contínua
)

di_slider = widgets.FloatSlider(
    value=40, min=0, max=100, step=5, 
    description="d_i (Firma 1):", 
    style={'description_width': 'initial'},
    continuous_update=True
)

dj_slider = widgets.FloatSlider(
    value=0, min=0, max=100, step=5, 
    description="d_j (Firma 2):", 
    style={'description_width': 'initial'},
    continuous_update=True
)

# ======================================================
# 🔹 Função de callback para os sliders
# ======================================================
def on_slider_change(change):
    update_plot(U_slider.value, di_slider.value, dj_slider.value)

# Conectar os sliders
U_slider.observe(on_slider_change, names='value')
di_slider.observe(on_slider_change, names='value')
dj_slider.observe(on_slider_change, names='value')

# ======================================================
# 🔹 Layout da interface
# ======================================================
sliders_box = widgets.VBox([
    widgets.HTML("<h3>Controles Interativos</h3>"),
    U_slider,
    di_slider, 
    dj_slider,
    widgets.HTML("<p><i>Altere os sliders para atualizar o gráfico dinamicamente</i></p>")
])

# Container principal
main_container = widgets.HBox([
    sliders_box, 
    fig
])

# ======================================================
# 🔹 Exibir interface e inicializar
# ======================================================
display(widgets.VBox([
    widgets.HTML("<h1>Modelo de Barganha de Nash - INTERATIVO</h1>"),
    main_container
]))

# Inicializar o gráfico
update_plot(U_slider.value, di_slider.value, dj_slider.value)